# Phase 5b — Closed-loop profit re-sim

Course-style month-by-month simulation with frozen **St_yours** rules from `params:profit`.

| Step | Action |
|------|--------|
| Each month | Build ABT from **approved_tx** pool → Gate B score → `apply_strategy` → grow pool |
| End | Attach defaults → P&L → compare to offline + `params:profit.reference` |

**Honesty:** same *method* as professor Process Simulation (feedback + P&L economics), not bit-identical SAS (our Gate B models; no Cross/PR). `reference` is the published course benchmark.

Helpers: `credit_scoring.profit` (not copied from `04`). Cut-offs live in `conf/base/parameters.yml`.

## §0 — Load data, params:profit, Gate B artifacts

In [12]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

from credit_scoring.profit.pnl import compute_pnl_table, filter_profit_window
from credit_scoring.profit.resim import run_closed_loop_resim
from credit_scoring.profit.rules import evaluate_strategy, rules_from_params
from credit_scoring.profit.scoring import score_abt_application

ROOT = Path("..").resolve()
DATA = ROOT / "data"
REPORTING = DATA / "08_reporting"
REPORTING.mkdir(parents=True, exist_ok=True)

with open(ROOT / "conf" / "base" / "parameters.yml") as f:
    _params = yaml.safe_load(f)

profit_params = _params["profit"]
behavioral_params = _params["behavioral"]
sim_params = _params["simulation"]
rules = rules_from_params(profit_params)

print("params:profit cutoffs / bad_customer / reference:")
display(pd.Series({
    "pd_css": rules["cutoffs"]["pd_css"],
    "pd_ins_high": rules["cutoffs"]["pd_ins_high"],
    "bad_feature": rules["bad_customer"]["feature"],
    "bad_threshold": rules["bad_customer"]["threshold"],
    "reference": profit_params["reference"],
    "window": f"{profit_params['window_start']}–{profit_params['window_end']}",
}))

params:profit cutoffs / bad_customer / reference:


pd_css                           0.4
pd_ins_high                 0.005329
bad_feature      agr12_Max_CMaxA_Due
bad_threshold                      3
reference                     731882
window                 197501–198712
dtype: object

In [13]:
def load_gate_b_artifacts(profit_params: dict, root: Path = ROOT):
    packages, points_tables, calibrations = {}, {}, {}
    for product, paths in profit_params["artifacts"].items():
        with open(root / paths["package"], "rb") as f:
            packages[product] = pickle.load(f)
        points_tables[product] = pd.read_parquet(root / paths["points"])
        with open(root / paths["calib"]) as f:
            calibrations[product] = json.load(f)
    return packages, points_tables, calibrations


production = pd.read_parquet(DATA / "02_intermediate" / "production.parquet")
transactions = pd.read_parquet(DATA / "02_intermediate" / "transactions.parquet")
default_df = pd.read_parquet(DATA / "02_intermediate" / "default.parquet")
packages, points_tables, calibrations = load_gate_b_artifacts(profit_params)

# --- checkpoint: load_resim_inputs ---
assert set(packages) == {"ins", "css"}
assert profit_params["cutoffs"]["pd_css"] is not None
print(
    f"checkpoint load_resim_inputs: production={production.shape} "
    f"transactions={transactions.shape} default={default_df.shape}"
)

checkpoint load_resim_inputs: production=(68644, 20) transactions=(1175025, 14) default=(68644, 8)


## §1 — Smoke: 3 months (197501–197503)

Pool must grow only on `decision=='A'`. Features come from `approved_tx`.

In [14]:
abt_smoke, dec_smoke = run_closed_loop_resim(
    production,
    transactions,
    default_df,
    behavioral_params,
    sim_params,
    profit_params,
    packages,
    points_tables,
    calibrations,
    start_period="197501",
    end_period="197503",
    verbose=True,
)

# --- checkpoint: smoke_resim ---
assert set(abt_smoke["period"].astype(str).unique()) <= {"197501", "197502", "197503"}
assert {"aid", "decision", "decline_reason", "pd"}.issubset(dec_smoke.columns)
assert dec_smoke["decision"].isin(["A", "D", "N"]).all()
n_a = int(dec_smoke["decision"].eq("A").sum())
print(f"checkpoint smoke_resim: apps={len(abt_smoke):,} decisions={len(dec_smoke):,} approved={n_a}")
display(dec_smoke["decline_reason"].value_counts())
display(dec_smoke.head())

197501 -> 197412:   306 apps,     3 approved, pool=  5,380
197502 -> 197501:   301 apps,     7 approved, pool=  5,535
197503 -> 197502:   315 apps,     4 approved, pool=  5,608
checkpoint smoke_resim: apps=922 decisions=922 approved=14


decline_reason
998 not active customer    476
2 PD cut-off on ins        432
999ok                       14
Name: count, dtype: int64

,cid,aid,product,period,decision,decline_reason,app_loan_amount,app_n_installments,pd
0,0000000625,css1975010100063,css,197501,N,998 not active customer,5000.0,24.0,0.152122
1,0000001330,css1975010100098,css,197501,N,998 not active customer,5000.0,24.0,0.152122
2,0000002179,css1975010100120,css,197501,N,998 not active customer,5000.0,24.0,0.152122
3,0000002366,css1975010100123,css,197501,N,998 not active customer,5000.0,24.0,0.175630
4,0000000079,css1975010200009,css,197501,N,998 not active customer,5000.0,24.0,0.162731


## §2 — Full window re-sim (197501–198712)

Expect several minutes. Re-run only when ready.

In [15]:
RUN_FULL = True  # set True for full 1975–87 (several minutes)

if RUN_FULL:
    abt_resim, decisions_resim = run_closed_loop_resim(
        production,
        transactions,
        default_df,
        behavioral_params,
        sim_params,
        profit_params,
        packages,
        points_tables,
        calibrations,
        start_period=profit_params["window_start"],
        end_period=profit_params["window_end"],
        verbose=True,
    )
    print(f"full resim: abt={abt_resim.shape} decisions={decisions_resim.shape}")
else:
    abt_resim, decisions_resim = abt_smoke, dec_smoke
    print("RUN_FULL=False — using smoke outputs for §3")

197501 -> 197412:   306 apps,     3 approved, pool=  5,380
197502 -> 197501:   301 apps,     7 approved, pool=  5,535
197503 -> 197502:   315 apps,     4 approved, pool=  5,608
197504 -> 197503:   308 apps,     4 approved, pool=  5,660
197505 -> 197504:   316 apps,    47 approved, pool=  6,481
197506 -> 197505:   317 apps,    15 approved, pool=  6,709
197507 -> 197506:   313 apps,     5 approved, pool=  6,800
197508 -> 197507:   314 apps,    14 approved, pool=  7,090
197509 -> 197508:   302 apps,    25 approved, pool=  7,612
197510 -> 197509:   319 apps,    14 approved, pool=  7,882
197511 -> 197510:   310 apps,     9 approved, pool=  8,045
197512 -> 197511:   379 apps,    16 approved, pool=  8,400
197601 -> 197512:   318 apps,    15 approved, pool=  8,644
197602 -> 197601:   311 apps,    11 approved, pool=  8,917
197603 -> 197602:   316 apps,     8 approved, pool=  9,076
197604 -> 197603:   316 apps,    10 approved, pool=  9,284
197605 -> 197604:   308 apps,    10 approved, pool=  9,4

## §3 — Score again for P&L, evaluate vs offline / reference

In [16]:
# Re-score closed-loop ABT (features already reflect strategy feedback)
scored_resim = score_abt_application(abt_resim, packages, points_tables, calibrations)
scored_pnl = compute_pnl_table(scored_resim, profit_params["economics"])
scored_window = filter_profit_window(
    scored_pnl, profit_params["window_start"], profit_params["window_end"]
)

# Attach strategy decisions (already computed in loop; re-merge for eval)
# Prefer loop decisions — they drove the pool
ev = evaluate_strategy(
    scored_window,
    decisions_resim,
    profit_params["window_start"],
    profit_params["window_end"],
)

offline_summary_path = REPORTING / "profit_summary.json"
offline_profit = None
if offline_summary_path.exists():
    with open(offline_summary_path) as f:
        offline_summary = json.load(f)
    offline_profit = offline_summary.get("offline_total_profit")

print("Closed-loop evaluation:")
display(pd.Series({k: v for k, v in ev.items() if k not in ("by_product", "by_year")}))
display(ev["by_product"])

print(f"offline_total_profit (Kedro): {offline_profit}")
print(f"closed_loop_total_profit:     {ev['total_profit']:,.2f}")
print(f"reference (params:profit):    {profit_params['reference']:,}")

Closed-loop evaluation:


total_profit     55535.009393
total_income    632455.409393
total_el        576920.400000
ar_ins               0.024072
ar_css               0.477086
bad_rate_ins         0.012433
bad_rate_css         0.281907
n_apps           49224.000000
n_accept          1381.000000
n_N              23885.000000
dtype: float64

,product,n_accept,total_profit,total_income,total_el,bad_rate
0,css,812,52685.156404,605435.156404,552750.0,0.281907
1,ins,569,2849.852988,27020.252988,24170.4,0.012433


offline_total_profit (Kedro): 1506547.208575406
closed_loop_total_profit:     55,535.01
reference (params:profit):    731,882


## §4 — Export

In [6]:
conclusion = {
    "best_strategy": "St_yours (bad + PD)",
    "source": "closed_loop",
    "cutoffs": rules["cutoffs"],
    "bad_customer": rules["bad_customer"],
    "closed_loop_total_profit": ev["total_profit"],
    "ar_ins": ev["ar_ins"],
    "ar_css": ev["ar_css"],
    "bad_rate_ins": ev["bad_rate_ins"],
    "bad_rate_css": ev["bad_rate_css"],
    "n_accept": ev["n_accept"],
    "offline_total_profit": offline_profit,
    "reference": profit_params["reference"],
    "window": [profit_params["window_start"], profit_params["window_end"]],
    "note": (
        "Course-style closed-loop with Gate B models + params:profit rules. "
        "reference is published course benchmark, not a re-run of SAS Process Simulation."
    ),
}

with open(REPORTING / "profit_resim_conclusion.json", "w") as f:
    json.dump(conclusion, f, indent=2)

decisions_resim.to_parquet(REPORTING / "decisions_resim.parquet", index=False)

by_loan = scored_window.merge(
    decisions_resim[["aid", "decision", "decline_reason"]].rename(
        columns={"decision": "strategy_decision", "decline_reason": "strategy_reason"}
    ),
    on="aid",
    how="left",
)
by_loan.to_parquet(REPORTING / "profit_resim_by_loan.parquet", index=False)

print("Wrote:")
print(" ", REPORTING / "profit_resim_conclusion.json")
print(" ", REPORTING / "decisions_resim.parquet")
print(" ", REPORTING / "profit_resim_by_loan.parquet")
display(pd.Series(conclusion))

Wrote:
  /Users/mac/Learningnewthings/credit-scoring/data/08_reporting/profit_resim_conclusion.json
  /Users/mac/Learningnewthings/credit-scoring/data/08_reporting/decisions_resim.parquet
  /Users/mac/Learningnewthings/credit-scoring/data/08_reporting/profit_resim_by_loan.parquet


best_strategy                                             St_yours (bad + PD)
source                                                            closed_loop
cutoffs                                 {'pd_css': 0.25, 'pd_ins_high': 0.01}
bad_customer                {'enabled': True, 'feature': 'agr12_Max_CMaxA_...
closed_loop_total_profit                                       -152550.569729
ar_ins                                                               0.156957
ar_css                                                               0.073414
bad_rate_ins                                                          0.02408
bad_rate_css                                                         0.345679
n_accept                                                                 3887
offline_total_profit                                           1506547.208575
reference                                                              731882
window                                                       [19